In [ ]:
OPENAI_API_KEY="{TOKEN}"
CYGNUS_JWT_TOKEN="{TOKEN}"
OPENAI_MODEL="gpt-4o-mini"
OPENAI_URL="https://api.openai.com/v1/chat/completions"

#### Imports all things which are needed

In [ ]:
import asyncio
import json
import urllib.request

In [ ]:
# ── OpenAI (stdlib HTTP, no SDK) ────────────────────────────────────────────────
def openai_reply(history: list[dict]):  # Returns a generator yielding strings
    body = json.dumps(
        {
            "model": OPENAI_MODEL,
            "messages": history,
            "max_tokens": 150,
            "temperature": 0.7,
            "stream": True,  # <-- Crucial: tells OpenAI to stream chunks
        }
    ).encode()

    req = urllib.request.Request(
        OPENAI_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
    )

    with urllib.request.urlopen(req, timeout=30) as r:
        for line in r:
            if not line:
                continue

            # Decode bytes and clean up whitespace
            line_str = line.decode("utf-8").strip()

            # OpenAI streams data using Server-Sent Events (SSE)
            if line_str.startswith("data: "):
                data_payload = line_str[6:]  # Strip out 'data: ' prefix

                # OpenAI signals the end of the stream with 'data: [DONE]'
                if data_payload == "[DONE]":
                    break

                try:
                    chunk = json.loads(data_payload)
                    # Safely navigate the nested dictionary for streaming
                    choices = chunk.get("choices", [])
                    if choices:
                        delta = choices[0].get("delta", {})
                        content = delta.get("content", "")
                        if content:
                            yield content
                except json.JSONDecodeError:
                    continue


In [ ]:
history = [
    {"role": "user", "content": "How Internet works Explain in 500 words"}
]

In [ ]:
gen = await asyncio.to_thread(openai_reply, history)

In [ ]:
def get_next_chunk(g):
    try:
        return next(g)
    except StopIteration:
        return None

In [ ]:
while True:
    await asyncio.sleep(0)

    chunk = await asyncio.to_thread(get_next_chunk, gen)
    if chunk is None:
        break

    print(chunk, end="")